In [ ]:
"""
ODIN Nexus v19.2: SOVEREIGN MULTI-AGENT STRATEGIC ADVISOR
Full-Stack Business Intelligence for Betopia Group (Odoo 19)

Senior Architecture:
━━━━━━━━━━━━━━━━━━━
Module 1: Sovereign Odoo 19 Bridge (Live XML-RPC — ALL Custom Models)
Module 2: Sales Pipeline Analyzer (sale.order + crm.lead pre-sale funnel)
Module 3: Operations Delivery Tracker (project.operation lifecycle)
Module 4: KPI Engine (employee.kpi + kpi.grade + kpi.level + bonus.calculation)
Module 5: Accounting & P&L Node (account.move + bank.statement.upload + budget)
Module 6: Altman Z-Score Solvency Benchmark (Private-firm Z'' variant)
Module 7: Monte Carlo Cash Runway (N=5000 stochastic simulations)
Module 8: GPT-4-Turbo Multi-Agent Oracle (Auditor + Financialist + Synthesizer)
Module 9: Conflict Validation (Legal vs Operations Paradox Detection)
Module 10: Sovereign Verdict PDF (Executive Heatmap + Impact Matrix)

DEPENDENCIES:
pip install langgraph langchain-openai reportlab openpyxl
"""

import os
import json
import logging
import operator
import random
import hashlib
import xmlrpc.client
from typing import Dict, Any, List, TypedDict, Annotated, Literal, Optional
from datetime import datetime, timedelta
from collections import defaultdict

# LangChain & LangGraph
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, PageBreak, KeepTogether, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger("ODIN-NEXUS-V19.2")

# ════════════════════════════════════════════════════════════════
# 1. ARCHITECTURAL STATE SCHEMA
# ════════════════════════════════════════════════════════════════

class SalesPipelineIntel(TypedDict):
    total_orders: int
    total_revenue: float
    paid_revenue: float
    unpaid_revenue: float
    collection_rate: float
    top_employees: List[Dict[str, Any]]
    top_customers: List[Dict[str, Any]]
    order_status_dist: Dict[str, int]
    payment_status_dist: Dict[str, int]
    monthly_trend: List[Dict[str, Any]]
    presale_funnel: Dict[str, Any]
    platform_breakdown: Dict[str, float]

class OperationsIntel(TypedDict):
    total_operations: int
    delivered_value: float
    pending_value: float
    delivery_rate: float
    status_distribution: Dict[str, int]
    team_performance: List[Dict[str, Any]]
    top_delivery_employees: List[Dict[str, Any]]

class KPIIntel(TypedDict):
    total_employees_tracked: int
    employees_meeting_target: int
    achievement_rate: float
    total_bonus_paid_bdt: float
    total_bonus_pending_bdt: float
    grade_distribution: Dict[str, int]
    role_distribution: Dict[str, int]
    top_performers: List[Dict[str, Any]]
    underperformers: List[Dict[str, Any]]
    penalty_count: int

class AccountingIntel(TypedDict):
    total_revenue_recognized: float
    total_expenses: float
    gross_profit: float
    gross_margin: float
    total_ar: float
    total_ap: float
    cash_balance: float
    total_assets: float
    total_liabilities: float
    equity: float
    retained_earnings: float
    budget_vs_actual: List[Dict[str, Any]]
    aging_ar: Dict[str, float]

class SolvencyIntel(TypedDict):
    z_score: float
    zone: str
    color_code: str
    component_breakdown: Dict[str, float]
    sensitivity_scenarios: List[Dict[str, Any]]

class CashRunwayIntel(TypedDict):
    burn_rate_monthly: float
    p5_days: int
    p50_days: int
    p95_days: int
    runway_months: float
    simulation_count: int

class CurrencyIntel(TypedDict):
    base_currency: str
    usd_to_bdt_rate: float
    bdt_to_usd_rate: float
    rate_date: str
    sales_revenue_usd: float
    sales_revenue_bdt: float
    bonus_total_bdt: float
    bonus_total_usd: float
    gross_profit_usd: float
    gross_profit_bdt: float
    bonus_to_revenue_pct: float
    currency_breakdown: List[Dict[str, Any]]

class HistoricalTrendIntel(TypedDict):
    yoy_revenue_growth: float
    yoy_orders_growth: float
    yoy_collection_rate_delta: float
    qoq_revenue_growth: float
    qoq_orders_growth: float
    prior_year_revenue: float
    prior_year_orders: int
    prior_year_collection_rate: float
    prior_quarter_revenue: float
    prior_quarter_orders: int

class NexusState(TypedDict):
    # Core Intelligence
    sales: SalesPipelineIntel
    operations: OperationsIntel
    kpi: KPIIntel
    accounting: AccountingIntel
    solvency: SolvencyIntel
    cash_runway: CashRunwayIntel
    strategic: StrategicIntel
    # Multi-Currency & Historical
    currency: CurrencyIntel
    historical: HistoricalTrendIntel
    # Governance
    proposed_directives: List[Dict[str, Any]]
    active_conflicts: List[str]
    authorized_advisory_ids: List[int]
    # Immutable Log
    audit_trail: Annotated[List[str], operator.add]
    status: Literal["SCANNING", "VALIDATING", "AWAITING_REVIEW", "FINALIZED"]
    data_source: str
    report_date: str

# ════════════════════════════════════════════════════════════════
# 2. SOVEREIGN ODOO 19 BRIDGE (ALL Custom Models)
# ════════════════════════════════════════════════════════════════

class OdooTransport(xmlrpc.client.SafeTransport):
    user_agent = 'ODIN-Nexus/19.2 (Sovereign Strategic Engine)'
    def send_user_agent(self, connection):
        connection.putheader("User-Agent", self.user_agent)

class NexusBridge:
    """
    Enterprise Read-Only Bridge to Odoo 19.
    Harvests data from ALL betopia_group_live custom models.
    Hard-locked: NO write operations permitted.
    """
    def __init__(self):
        self.url = os.environ.get("ODOO_URL", "").rstrip('/')
        self.db = os.environ.get("ODOO_DB", "")
        self.username = os.environ.get("ODOO_USERNAME", "")
        self.password = os.environ.get("ODOO_API_KEY", "")
        self.transport = OdooTransport()
        self._uid = None
        self._models = None

    def _authenticate(self):
        if self._uid:
            return self._uid
        common = xmlrpc.client.ServerProxy(f'{self.url}/xmlrpc/2/common', transport=self.transport)
        self._uid = common.authenticate(self.db, self.username, self.password, {})
        if not self._uid:
            raise ConnectionError("Odoo Authentication Failed — check ODOO_USERNAME & ODOO_API_KEY env vars")
        self._models = xmlrpc.client.ServerProxy(f'{self.url}/xmlrpc/2/object', transport=self.transport)
        logger.info(f"Authenticated to {self.url} as UID {self._uid}")
        return self._uid

    def _search_read(self, model: str, domain: list, fields: list, limit: int = 0, order: str = "") -> list:
        uid = self._authenticate()
        kwargs = {'fields': fields}
        if limit:
            kwargs['limit'] = limit
        if order:
            kwargs['order'] = order
        return self._models.execute_kw(self.db, uid, self.password, model, 'search_read', [domain], kwargs)

    def _search_count(self, model: str, domain: list) -> int:
        uid = self._authenticate()
        return self._models.execute_kw(self.db, uid, self.password, model, 'search_count', [domain])

    def _read_group(self, model: str, domain: list, fields: list, groupby: list) -> list:
        uid = self._authenticate()
        return self._models.execute_kw(self.db, uid, self.password, model, 'read_group', [domain, fields, groupby])

    # ── SALES PIPELINE ──────────────────────────────────────────

    def fetch_sales_data(self, date_from: str, date_to: str) -> SalesPipelineIntel:
        """Harvest sale.order with KPI-relevant fields."""
        logger.info(f"Fetching sales data: {date_from} → {date_to}")

        base_domain = [('date_order', '>=', date_from), ('date_order', '<=', date_to)]
        so_fields = [
            'name', 'amount_total', 'state', 'payment_state', 'payment_date',
            'partner_id', 'sales_employee_id', 'company_id', 'date_order',
            'order_status', 'currency_id'
        ]

        all_orders = self._search_read('sale.order', base_domain, so_fields)

        total_revenue = sum(o['amount_total'] for o in all_orders)
        paid_revenue = sum(o['amount_total'] for o in all_orders if o.get('payment_state') == 'paid')
        unpaid_revenue = total_revenue - paid_revenue
        collection_rate = (paid_revenue / total_revenue * 100) if total_revenue > 0 else 0

        # Order status distribution (coerce False → 'unknown' to avoid mixed-type dict keys)
        status_dist = defaultdict(int)
        for o in all_orders:
            status_dist[o.get('order_status') or o.get('state') or 'unknown'] += 1

        # Payment status distribution
        payment_dist = defaultdict(int)
        for o in all_orders:
            payment_dist[o.get('payment_state') or 'unknown'] += 1

        # Top employees by revenue
        emp_revenue = defaultdict(lambda: {"name": "", "total": 0, "paid": 0, "count": 0})
        for o in all_orders:
            emp = o.get('sales_employee_id')
            if emp:
                eid = emp[0] if isinstance(emp, (list, tuple)) else emp
                ename = emp[1] if isinstance(emp, (list, tuple)) else str(emp)
                emp_revenue[eid]["name"] = ename
                emp_revenue[eid]["total"] += o['amount_total']
                emp_revenue[eid]["count"] += 1
                if o.get('payment_state') == 'paid':
                    emp_revenue[eid]["paid"] += o['amount_total']

        top_employees = sorted(emp_revenue.values(), key=lambda x: x['total'], reverse=True)[:10]

        # Top customers by revenue
        cust_revenue = defaultdict(lambda: {"name": "", "total": 0, "count": 0})
        for o in all_orders:
            partner = o.get('partner_id')
            if partner:
                pid = partner[0] if isinstance(partner, (list, tuple)) else partner
                pname = partner[1] if isinstance(partner, (list, tuple)) else str(partner)
                cust_revenue[pid]["name"] = pname
                cust_revenue[pid]["total"] += o['amount_total']
                cust_revenue[pid]["count"] += 1

        top_customers = sorted(cust_revenue.values(), key=lambda x: x['total'], reverse=True)[:10]

        # Monthly trend
        monthly = defaultdict(lambda: {"revenue": 0, "paid": 0, "count": 0})
        for o in all_orders:
            month_key = o['date_order'][:7] if o.get('date_order') else 'unknown'
            monthly[month_key]["revenue"] += o['amount_total']
            monthly[month_key]["count"] += 1
            if o.get('payment_state') == 'paid':
                monthly[month_key]["paid"] += o['amount_total']

        monthly_trend = [{"month": k, **v} for k, v in sorted(monthly.items())]

        # Platform breakdown (from presale → sale linkage)
        platform_breakdown = {}
        try:
            presales = self._search_read('crm.lead', [
                ('is_presale', '=', True), ('sales_status', '=', 'sold'),
                ('sold_date', '>=', date_from), ('sold_date', '<=', date_to)
            ], ['platform_source_id', 'sold_amount'])
            plat_rev = defaultdict(float)
            for ps in presales:
                src = ps.get('platform_source_id')
                if src:
                    plat_name = src[1] if isinstance(src, (list, tuple)) else str(src)
                    plat_rev[plat_name] += ps.get('sold_amount', 0) or 0
            platform_breakdown = dict(plat_rev)
        except Exception:
            logger.warning("Presale platform data unavailable")

        # Presale funnel
        presale_funnel = {}
        try:
            total_queries = self._search_count('crm.lead', [('is_presale', '=', True), ('create_date', '>=', date_from)])
            won_queries = self._search_count('crm.lead', [('is_presale', '=', True), ('sales_status', '=', 'sold'), ('create_date', '>=', date_from)])
            lost_queries = self._search_count('crm.lead', [('is_presale', '=', True), ('sales_status', '=', 'not_sold'), ('active', '=', False), ('create_date', '>=', date_from)])
            presale_funnel = {
                "total_queries": total_queries,
                "converted": won_queries,
                "lost": lost_queries,
                "conversion_rate": round(won_queries / max(total_queries, 1) * 100, 1)
            }
        except Exception:
            logger.warning("Presale funnel data unavailable")

        return {
            "total_orders": len(all_orders),
            "total_revenue": round(total_revenue, 2),
            "paid_revenue": round(paid_revenue, 2),
            "unpaid_revenue": round(unpaid_revenue, 2),
            "collection_rate": round(collection_rate, 1),
            "top_employees": top_employees,
            "top_customers": top_customers,
            "order_status_dist": dict(status_dist),
            "payment_status_dist": dict(payment_dist),
            "monthly_trend": monthly_trend,
            "presale_funnel": presale_funnel,
            "platform_breakdown": platform_breakdown
        }

    # ── OPERATIONS ───────────────────────────────────────────────

    def fetch_operations_data(self, date_from: str, date_to: str) -> OperationsIntel:
        """Harvest project.operation delivery data."""
        logger.info("Fetching operations data")

        ops = self._search_read('project.operation', [
            ('date', '>=', date_from), ('date', '<=', date_to)
        ], [
            'name', 'employee_id', 'assign_employee_id', 'monetary_value',
            'order_status', 'sale_status', 'payment_state', 'payment_date',
            'assigned_team_id', 'service_type_id', 'company_id'
        ])

        total_ops = len(ops)
        delivered_value = sum(o['monetary_value'] or 0 for o in ops if o.get('order_status') in ('delivered', 'complete'))
        pending_value = sum(o['monetary_value'] or 0 for o in ops if o.get('order_status') in ('nra', 'wip'))
        delivery_rate = (delivered_value / (delivered_value + pending_value) * 100) if (delivered_value + pending_value) > 0 else 0

        # Status distribution (coerce False → 'unknown')
        status_dist = defaultdict(int)
        for o in ops:
            status_dist[o.get('order_status') or 'unknown'] += 1

        # Team performance
        team_perf = defaultdict(lambda: {"name": "", "delivered": 0, "total": 0, "count": 0})
        for o in ops:
            team = o.get('assigned_team_id')
            if team:
                tid = team[0] if isinstance(team, (list, tuple)) else team
                tname = team[1] if isinstance(team, (list, tuple)) else str(team)
                team_perf[tid]["name"] = tname
                team_perf[tid]["total"] += o.get('monetary_value', 0) or 0
                team_perf[tid]["count"] += 1
                if o.get('order_status') in ('delivered', 'complete'):
                    team_perf[tid]["delivered"] += o.get('monetary_value', 0) or 0

        # Top delivery employees
        emp_delivery = defaultdict(lambda: {"name": "", "delivered_value": 0, "count": 0})
        for o in ops:
            emp = o.get('assign_employee_id') or o.get('employee_id')
            if emp:
                eid = emp[0] if isinstance(emp, (list, tuple)) else emp
                ename = emp[1] if isinstance(emp, (list, tuple)) else str(emp)
                emp_delivery[eid]["name"] = ename
                emp_delivery[eid]["count"] += 1
                if o.get('order_status') in ('delivered', 'complete'):
                    emp_delivery[eid]["delivered_value"] += o.get('monetary_value', 0) or 0

        return {
            "total_operations": total_ops,
            "delivered_value": round(delivered_value, 2),
            "pending_value": round(pending_value, 2),
            "delivery_rate": round(delivery_rate, 1),
            "status_distribution": dict(status_dist),
            "team_performance": sorted(team_perf.values(), key=lambda x: x['delivered'], reverse=True)[:10],
            "top_delivery_employees": sorted(emp_delivery.values(), key=lambda x: x['delivered_value'], reverse=True)[:10]
        }

    # ── KPI & BONUS ──────────────────────────────────────────────

    def fetch_kpi_data(self, year: int, quarter: int = 0) -> KPIIntel:
        """Harvest employee.kpi, kpi.grade, kpi.level, bonus.calculation."""
        logger.info(f"Fetching KPI data: year={year}, quarter={quarter}")

        kpi_domain = [('year', '=', str(year))]
        if quarter:
            kpi_domain.append(('quarter', '=', str(quarter)))

        kpis = self._search_read('employee.kpi', kpi_domain, [
            'employee_id', 'role_id', 'role_type', 'grade_id', 'minimum_target',
            'total_paid', 'total_unpaid', 'total_sales', 'bonus_amount',
            'carry_paid_sales', 'shortfall_amount', 'is_penalty', 'state',
            'period_start', 'period_end', 'company_id'
        ])

        total_tracked = len(set(k['employee_id'][0] for k in kpis if k.get('employee_id')))
        meeting_target = len([k for k in kpis if k.get('total_paid', 0) >= k.get('minimum_target', 0) and k.get('minimum_target', 0) > 0])
        achievement_rate = (meeting_target / max(len(kpis), 1)) * 100
        total_bonus_paid = sum(k.get('bonus_amount', 0) or 0 for k in kpis if k.get('state') == 'paid')
        total_bonus_pending = sum(k.get('bonus_amount', 0) or 0 for k in kpis if k.get('state') in ('draft', 'confirmed'))

        # Grade distribution
        grade_dist = defaultdict(int)
        for k in kpis:
            g = k.get('grade_id')
            gname = g[1] if isinstance(g, (list, tuple)) else str(g or 'Ungraded')
            grade_dist[gname] += 1

        # Role distribution (coerce False → 'unknown')
        role_dist = defaultdict(int)
        for k in kpis:
            role_dist[k.get('role_type') or 'unknown'] += 1

        # Top performers by total_paid
        emp_kpi = defaultdict(lambda: {"name": "", "total_paid": 0, "target": 0, "bonus": 0, "role": ""})
        for k in kpis:
            emp = k.get('employee_id')
            if emp:
                eid = emp[0] if isinstance(emp, (list, tuple)) else emp
                ename = emp[1] if isinstance(emp, (list, tuple)) else str(emp)
                emp_kpi[eid]["name"] = ename
                emp_kpi[eid]["total_paid"] += k.get('total_paid', 0) or 0
                emp_kpi[eid]["target"] = k.get('minimum_target', 0) or 0
                emp_kpi[eid]["bonus"] += k.get('bonus_amount', 0) or 0
                emp_kpi[eid]["role"] = k.get('role_type') or ''

        sorted_performers = sorted(emp_kpi.values(), key=lambda x: x['total_paid'], reverse=True)

        penalty_count = len([k for k in kpis if k.get('is_penalty')])

        return {
            "total_employees_tracked": total_tracked,
            "employees_meeting_target": meeting_target,
            "achievement_rate": round(achievement_rate, 1),
            "total_bonus_paid_bdt": round(total_bonus_paid, 2),
            "total_bonus_pending_bdt": round(total_bonus_pending, 2),
            "grade_distribution": dict(grade_dist),
            "role_distribution": dict(role_dist),
            "top_performers": sorted_performers[:10],
            "underperformers": [p for p in sorted_performers if p['total_paid'] < p['target']][:10],
            "penalty_count": penalty_count
        }

    # ── ACCOUNTING & P&L ─────────────────────────────────────────

    def fetch_accounting_data(self, date_from: str, date_to: str) -> AccountingIntel:
        """Harvest account.account balances, account.move, budget data."""
        logger.info("Fetching accounting data")

        # Get account balance field name
        try:
            acc_fields = self._models.execute_kw(
                self.db, self._uid, self.password,
                'account.account', 'fields_get', [], {'attributes': ['string']}
            )
            balance_field = 'current_balance' if 'current_balance' in acc_fields else 'balance'
        except Exception:
            balance_field = 'balance'

        accounts = self._search_read('account.account', [], ['name', balance_field, 'code', 'account_type'])

        # Classify by account code prefix (Bangladeshi CoA)
        # Note: Odoo returns False instead of None/'' for empty fields
        cash = sum(abs(a[balance_field]) for a in accounts if any(kw in (a.get('name') or '').lower() for kw in ['cash', 'bank']))
        ar = sum(abs(a[balance_field]) for a in accounts if 'receivable' in (a.get('account_type') or '') or (a.get('code') or '').startswith('12'))
        ap = sum(abs(a[balance_field]) for a in accounts if 'payable' in (a.get('account_type') or '') or (a.get('code') or '').startswith('21'))
        assets = sum(abs(a[balance_field]) for a in accounts if (a.get('code') or '').startswith('1'))
        liabilities = sum(abs(a[balance_field]) for a in accounts if (a.get('code') or '').startswith('2'))
        equity = sum(abs(a[balance_field]) for a in accounts if (a.get('code') or '').startswith('3'))
        retained = sum(abs(a[balance_field]) for a in accounts if 'retained' in (a.get('name') or '').lower())

        # Revenue from posted invoices in period
        revenue_invoices = self._search_read('account.move', [
            ('move_type', '=', 'out_invoice'), ('state', '=', 'posted'),
            ('invoice_date', '>=', date_from), ('invoice_date', '<=', date_to)
        ], ['amount_total_signed', 'payment_state', 'partner_id', 'invoice_date_due'])

        total_revenue_rec = sum(abs(inv['amount_total_signed']) for inv in revenue_invoices)

        # Expenses from posted bills
        expense_bills = self._search_read('account.move', [
            ('move_type', '=', 'in_invoice'), ('state', '=', 'posted'),
            ('invoice_date', '>=', date_from), ('invoice_date', '<=', date_to)
        ], ['amount_total_signed'])

        total_expenses = sum(abs(b['amount_total_signed']) for b in expense_bills)

        gross_profit = total_revenue_rec - total_expenses
        gross_margin = (gross_profit / max(total_revenue_rec, 1)) * 100

        # AR Aging
        today = datetime.now().date()
        aging = {"current": 0, "30_days": 0, "60_days": 0, "90_days": 0, "over_90": 0}
        for inv in revenue_invoices:
            if inv.get('payment_state') not in ('paid', 'reversed'):
                due = inv.get('invoice_date_due')
                if due:
                    try:
                        due_date = datetime.strptime(due, '%Y-%m-%d').date() if isinstance(due, str) else due
                        days_overdue = (today - due_date).days
                        amt = abs(inv['amount_total_signed'])
                        if days_overdue <= 0:
                            aging["current"] += amt
                        elif days_overdue <= 30:
                            aging["30_days"] += amt
                        elif days_overdue <= 60:
                            aging["60_days"] += amt
                        elif days_overdue <= 90:
                            aging["90_days"] += amt
                        else:
                            aging["over_90"] += amt
                    except Exception:
                        aging["current"] += abs(inv['amount_total_signed'])

        # Budget vs Actual
        budget_data = []
        try:
            budgets = self._search_read('crossovered.budget.lines', [
                ('date_from', '>=', date_from), ('date_to', '<=', date_to),
                ('crossovered_budget_state', '=', 'validate')
            ], ['general_budget_id', 'planned_amount', 'practical_amount', 'percentage'])

            for b in budgets:
                budget_data.append({
                    "category": b['general_budget_id'][1] if isinstance(b['general_budget_id'], (list, tuple)) else str(b['general_budget_id']),
                    "planned": b.get('planned_amount', 0),
                    "actual": b.get('practical_amount', 0),
                    "variance_pct": b.get('percentage', 0)
                })
        except Exception:
            logger.warning("Budget data unavailable")

        return {
            "total_revenue_recognized": round(total_revenue_rec, 2),
            "total_expenses": round(total_expenses, 2),
            "gross_profit": round(gross_profit, 2),
            "gross_margin": round(gross_margin, 1),
            "total_ar": round(ar, 2),
            "total_ap": round(ap, 2),
            "cash_balance": round(cash, 2),
            "total_assets": round(max(assets, 1.0), 2),
            "total_liabilities": round(max(liabilities, 1.0), 2),
            "equity": round(equity, 2),
            "retained_earnings": round(retained if retained > 0 else equity * 0.3, 2),
            "budget_vs_actual": budget_data,
            "aging_ar": {k: round(v, 2) for k, v in aging.items()}
        }

    # ── CURRENCY RATES ───────────────────────────────────────────

    def fetch_currency_rates(self) -> Dict[str, Any]:
        """Harvest live exchange rates from res.currency + res.currency.rate."""
        logger.info("Fetching currency exchange rates")

        currencies = self._search_read('res.currency', [
            ('name', 'in', ['USD', 'BDT']), ('active', '=', True)
        ], ['name', 'rate', 'symbol', 'rounding'])

        usd_rate = 1.0
        bdt_rate = 1.0
        for c in currencies:
            if c['name'] == 'USD':
                usd_rate = c.get('rate', 1.0) or 1.0
            elif c['name'] == 'BDT':
                bdt_rate = c.get('rate', 1.0) or 1.0

        # rate is relative to company currency; compute cross rate
        # If company currency is BDT: USD rate = X means 1 BDT = X USD, BDT rate = 1
        # If company currency is USD: BDT rate = X means 1 USD = X BDT
        if bdt_rate > 0 and usd_rate > 0:
            usd_to_bdt = bdt_rate / usd_rate
            bdt_to_usd = usd_rate / bdt_rate
        else:
            usd_to_bdt = 120.0  # Fallback BDT/USD rate
            bdt_to_usd = 1.0 / 120.0

        # Sanity check: BDT/USD should be 100-150 range
        if usd_to_bdt < 10 or usd_to_bdt > 500:
            logger.warning(f"Unusual USD→BDT rate {usd_to_bdt:.2f}, using fallback 120.0")
            usd_to_bdt = 120.0
            bdt_to_usd = 1.0 / 120.0

        # Get sale.order currency breakdown
        currency_breakdown = []
        try:
            groups = self._read_group('sale.order', [
                ('date_order', '>=', datetime.now().replace(month=1, day=1).strftime('%Y-%m-%d'))
            ], ['currency_id', 'amount_total'], ['currency_id'])
            for g in groups:
                cname = g.get('currency_id')
                if cname:
                    cname = cname[1] if isinstance(cname, (list, tuple)) else str(cname)
                else:
                    cname = 'Unknown'
                currency_breakdown.append({
                    "currency": cname,
                    "total": g.get('amount_total', 0) or 0,
                    "count": g.get('currency_id_count', 0) or 0
                })
        except Exception:
            logger.warning("Currency breakdown unavailable")

        return {
            "usd_to_bdt": round(usd_to_bdt, 2),
            "bdt_to_usd": round(bdt_to_usd, 6),
            "rate_date": datetime.now().strftime('%Y-%m-%d'),
            "currency_breakdown": currency_breakdown
        }

    # ── HISTORICAL TREND DATA ────────────────────────────────────

    def fetch_historical_sales(self, date_from: str, date_to: str) -> Dict[str, Any]:
        """Fetch aggregate sales data for a historical period."""
        orders = self._search_read('sale.order', [
            ('date_order', '>=', date_from), ('date_order', '<=', date_to)
        ], ['amount_total', 'payment_state'])
        total_revenue = sum(o['amount_total'] for o in orders)
        paid_revenue = sum(o['amount_total'] for o in orders if o.get('payment_state') == 'paid')
        collection_rate = (paid_revenue / max(total_revenue, 1)) * 100
        return {
            "revenue": round(total_revenue, 2),
            "orders": len(orders),
            "paid_revenue": round(paid_revenue, 2),
            "collection_rate": round(collection_rate, 1)
        }

    def fetch_historical_operations(self, date_from: str, date_to: str) -> Dict[str, Any]:
        """Fetch aggregate operations data for a historical period."""
        ops = self._search_read('project.operation', [
            ('date', '>=', date_from), ('date', '<=', date_to)
        ], ['monetary_value', 'order_status'])
        delivered = sum(o['monetary_value'] or 0 for o in ops if o.get('order_status') in ('delivered', 'complete'))
        total_val = sum(o['monetary_value'] or 0 for o in ops)
        delivery_rate = (delivered / max(total_val, 1)) * 100
        return {
            "total": len(ops),
            "delivered_value": round(delivered, 2),
            "delivery_rate": round(delivery_rate, 1)
        }

    def fetch_historical_kpi(self, year: int) -> Dict[str, Any]:
        """Fetch aggregate KPI achievement for a historical year."""
        kpis = self._search_read('employee.kpi', [
            ('year', '=', str(year))
        ], ['total_paid', 'minimum_target'])
        meeting = len([k for k in kpis if (k.get('total_paid', 0) or 0) >= (k.get('minimum_target', 0) or 0) and (k.get('minimum_target', 0) or 0) > 0])
        achievement = (meeting / max(len(kpis), 1)) * 100
        return {
            "total_kpi_records": len(kpis),
            "achievement_rate": round(achievement, 1)
        }

# ════════════════════════════════════════════════════════════════
# 3. INTELLIGENCE NODES
# ════════════════════════════════════════════════════════════════

def _get_bridge() -> NexusBridge:
    bridge = NexusBridge()
    bridge._authenticate()
    return bridge

def _get_period(state: NexusState):
    today = datetime.now()
    date_from = today.replace(month=1, day=1).strftime('%Y-%m-%d')
    date_to = today.strftime('%Y-%m-%d')
    return date_from, date_to, today

# ── Node 1: Sales Pipeline ──────────────────────────────────────

def sales_pipeline_node(state: NexusState) -> Dict:
    """Harvests complete sales pipeline from sale.order + crm.lead."""
    bridge = _get_bridge()
    date_from, date_to, today = _get_period(state)

    try:
        sales_data = bridge.fetch_sales_data(date_from, date_to)
    except Exception as e:
        logger.error(f"Sales node failed: {e}")
        sales_data = {
            "total_orders": 0, "total_revenue": 0, "paid_revenue": 0,
            "unpaid_revenue": 0, "collection_rate": 0, "top_employees": [],
            "top_customers": [], "order_status_dist": {}, "payment_status_dist": {},
            "monthly_trend": [], "presale_funnel": {}, "platform_breakdown": {}
        }

    return {
        "sales": sales_data,
        "data_source": f"LIVE ODOO 19 ({today.strftime('%Y-%m-%d %H:%M')})",
        "report_date": today.strftime('%Y-%m-%d'),
        "audit_trail": [f"Sales Node: {sales_data['total_orders']} orders, ${sales_data['total_revenue']:,.2f} revenue harvested."]
    }

# ── Node 2: Operations Delivery ─────────────────────────────────

def operations_node(state: NexusState) -> Dict:
    """Harvests project.operation delivery lifecycle."""
    bridge = _get_bridge()
    date_from, date_to, _ = _get_period(state)

    try:
        ops_data = bridge.fetch_operations_data(date_from, date_to)
    except Exception as e:
        logger.error(f"Operations node failed: {e}")
        ops_data = {
            "total_operations": 0, "delivered_value": 0, "pending_value": 0,
            "delivery_rate": 0, "status_distribution": {}, "team_performance": [],
            "top_delivery_employees": []
        }

    return {
        "operations": ops_data,
        "audit_trail": [f"Operations Node: {ops_data['total_operations']} operations, ${ops_data['delivered_value']:,.2f} delivered."]
    }

# ── Node 3: KPI Engine ──────────────────────────────────────────

def kpi_engine_node(state: NexusState) -> Dict:
    """Harvests employee.kpi + bonus.calculation data."""
    bridge = _get_bridge()
    today = datetime.now()

    try:
        kpi_data = bridge.fetch_kpi_data(today.year)
    except Exception as e:
        logger.error(f"KPI node failed: {e}")
        kpi_data = {
            "total_employees_tracked": 0, "employees_meeting_target": 0,
            "achievement_rate": 0, "total_bonus_paid_bdt": 0,
            "total_bonus_pending_bdt": 0, "grade_distribution": {},
            "role_distribution": {}, "top_performers": [],
            "underperformers": [], "penalty_count": 0
        }

    return {
        "kpi": kpi_data,
        "audit_trail": [f"KPI Node: {kpi_data['total_employees_tracked']} employees tracked, {kpi_data['achievement_rate']}% achievement rate."]
    }

# ── Node 4: Accounting & P&L ────────────────────────────────────

def accounting_node(state: NexusState) -> Dict:
    """Harvests account.account, account.move, budget data."""
    bridge = _get_bridge()
    date_from, date_to, _ = _get_period(state)

    try:
        acct_data = bridge.fetch_accounting_data(date_from, date_to)
    except Exception as e:
        logger.error(f"Accounting node failed: {e}")
        acct_data = {
            "total_revenue_recognized": 0, "total_expenses": 0, "gross_profit": 0,
            "gross_margin": 0, "total_ar": 0, "total_ap": 0, "cash_balance": 0,
            "total_assets": 1, "total_liabilities": 1, "equity": 0,
            "retained_earnings": 0, "budget_vs_actual": [], "aging_ar": {}
        }

    return {
        "accounting": acct_data,
        "audit_trail": [f"Accounting Node: P&L harvested — Gross Profit ${acct_data['gross_profit']:,.2f} ({acct_data['gross_margin']}% margin)."]
    }

# ── Node 4b: Currency Normalization ──────────────────────────────

def currency_normalization_node(state: NexusState) -> Dict:
    """Normalizes all monetary values into both USD and BDT using live exchange rates."""
    bridge = _get_bridge()

    try:
        fx = bridge.fetch_currency_rates()
    except Exception as e:
        logger.error(f"Currency node failed: {e}")
        fx = {"usd_to_bdt": 120.0, "bdt_to_usd": 1/120.0, "rate_date": datetime.now().strftime('%Y-%m-%d'), "currency_breakdown": []}

    usd_to_bdt = fx['usd_to_bdt']
    bdt_to_usd = fx['bdt_to_usd']

    sales_usd = state['sales']['total_revenue']
    sales_bdt = round(sales_usd * usd_to_bdt, 2)
    bonus_bdt = state['kpi']['total_bonus_paid_bdt']
    bonus_usd = round(bonus_bdt * bdt_to_usd, 2)
    gp_usd = state['accounting']['gross_profit']
    gp_bdt = round(gp_usd * usd_to_bdt, 2)

    # Bonus as % of revenue (both in USD for apples-to-apples)
    bonus_to_rev = round(bonus_usd / max(sales_usd, 1) * 100, 2)

    return {
        "currency": {
            "base_currency": "USD",
            "usd_to_bdt_rate": usd_to_bdt,
            "bdt_to_usd_rate": bdt_to_usd,
            "rate_date": fx['rate_date'],
            "sales_revenue_usd": sales_usd,
            "sales_revenue_bdt": sales_bdt,
            "bonus_total_bdt": bonus_bdt,
            "bonus_total_usd": bonus_usd,
            "gross_profit_usd": gp_usd,
            "gross_profit_bdt": gp_bdt,
            "bonus_to_revenue_pct": bonus_to_rev,
            "currency_breakdown": fx['currency_breakdown']
        },
        "audit_trail": [f"Currency Node: USD→BDT {usd_to_bdt:.2f} | Revenue ৳{sales_bdt:,.0f} | Bonus ${bonus_usd:,.2f} ({bonus_to_rev}% of revenue)."]
    }

# ── Node 4c: Historical Trend Comparison (YoY/QoQ) ──────────────

def historical_trend_node(state: NexusState) -> Dict:
    """Compares current period vs prior year and prior quarter."""
    bridge = _get_bridge()
    today = datetime.now()

    # Current period: Jan 1 → today
    cur_from = today.replace(month=1, day=1).strftime('%Y-%m-%d')
    cur_to = today.strftime('%Y-%m-%d')

    # Prior year same period
    py_from = today.replace(year=today.year - 1, month=1, day=1).strftime('%Y-%m-%d')
    py_to = today.replace(year=today.year - 1).strftime('%Y-%m-%d')

    # Current quarter boundaries
    q = (today.month - 1) // 3 + 1
    cq_start = datetime(today.year, (q - 1) * 3 + 1, 1)
    cq_end = today

    # Prior quarter
    if q == 1:
        pq_start = datetime(today.year - 1, 10, 1)
        pq_end = datetime(today.year - 1, 12, 31)
    else:
        pq_start = datetime(today.year, (q - 2) * 3 + 1, 1)
        pq_end = cq_start - timedelta(days=1)

    trend_signals = []

    try:
        # YoY Sales
        py_sales = bridge.fetch_historical_sales(py_from, py_to)
        cur_sales_rev = state['sales']['total_revenue']
        cur_orders = state['sales']['total_orders']
        cur_collection = state['sales']['collection_rate']

        yoy_rev = ((cur_sales_rev - py_sales['revenue']) / max(py_sales['revenue'], 1)) * 100
        yoy_ord = ((cur_orders - py_sales['orders']) / max(py_sales['orders'], 1)) * 100
        yoy_coll = cur_collection - py_sales['collection_rate']

        if yoy_rev > 20:
            trend_signals.append(f"STRONG GROWTH: Revenue up {yoy_rev:.0f}% YoY")
        elif yoy_rev < -10:
            trend_signals.append(f"REVENUE DECLINE: Down {abs(yoy_rev):.0f}% YoY — investigate")
        if yoy_coll < -10:
            trend_signals.append(f"COLLECTION DETERIORATION: Down {abs(yoy_coll):.1f}pp YoY")

        # YoY Operations
        py_ops = bridge.fetch_historical_operations(py_from, py_to)
        yoy_delivery = state['operations']['delivery_rate'] - py_ops['delivery_rate']

        # YoY KPI
        py_kpi = bridge.fetch_historical_kpi(today.year - 1)
        yoy_kpi = state['kpi']['achievement_rate'] - py_kpi['achievement_rate']
        if yoy_kpi < -15:
            trend_signals.append(f"KPI COLLAPSE: Achievement down {abs(yoy_kpi):.1f}pp YoY")

        # QoQ Sales
        pq_sales = bridge.fetch_historical_sales(pq_start.strftime('%Y-%m-%d'), pq_end.strftime('%Y-%m-%d'))
        cq_sales = bridge.fetch_historical_sales(cq_start.strftime('%Y-%m-%d'), cq_end.strftime('%Y-%m-%d'))
        qoq_rev = ((cq_sales['revenue'] - pq_sales['revenue']) / max(pq_sales['revenue'], 1)) * 100
        qoq_ord = ((cq_sales['orders'] - pq_sales['orders']) / max(pq_sales['orders'], 1)) * 100

        if qoq_rev < -20:
            trend_signals.append(f"QUARTERLY SLUMP: Revenue down {abs(qoq_rev):.0f}% QoQ")

        # Build quarterly trend (last 4 quarters)
        quarterly_trend = []
        for qi in range(4):
            offset_year = today.year if q - qi > 0 else today.year - 1
            offset_q = q - qi if q - qi > 0 else q - qi + 4
            qs = datetime(offset_year, (offset_q - 1) * 3 + 1, 1)
            if offset_q == 4:
                qe = datetime(offset_year, 12, 31)
            else:
                qe = datetime(offset_year, offset_q * 3 + 1, 1) - timedelta(days=1)
            qe = min(qe, today)
            if qs > today:
                continue
            q_data = bridge.fetch_historical_sales(qs.strftime('%Y-%m-%d'), qe.strftime('%Y-%m-%d'))
            quarterly_trend.append({
                "quarter": f"Q{offset_q} {offset_year}",
                "revenue": q_data['revenue'],
                "orders": q_data['orders'],
                "collection_rate": q_data['collection_rate']
            })
        quarterly_trend.reverse()

        if not trend_signals:
            trend_signals.append("STABLE: No significant YoY/QoQ anomalies detected")

    except Exception as e:
        logger.error(f"Historical trend node failed: {e}")
        yoy_rev = yoy_ord = yoy_coll = yoy_delivery = yoy_kpi = 0.0
        qoq_rev = qoq_ord = 0.0
        py_sales = {"revenue": 0, "orders": 0}
        pq_sales = {"revenue": 0, "orders": 0}
        quarterly_trend = []
        trend_signals = ["Historical data unavailable — first-year deployment"]

    return {
        "historical": {
            "yoy_revenue_growth": round(yoy_rev, 1),
            "yoy_orders_growth": round(yoy_ord, 1),
            "yoy_collection_rate_delta": round(yoy_coll, 1),
            "yoy_delivery_rate_delta": round(yoy_delivery, 1),
            "yoy_kpi_achievement_delta": round(yoy_kpi, 1),
            "qoq_revenue_growth": round(qoq_rev, 1),
            "qoq_orders_growth": round(qoq_ord, 1),
            "prior_year_revenue": py_sales['revenue'],
            "prior_year_orders": py_sales['orders'],
            "prior_quarter_revenue": pq_sales['revenue'],
            "prior_quarter_orders": pq_sales['orders'],
            "quarterly_trend": quarterly_trend,
            "trend_signals": trend_signals
        },
        "audit_trail": [f"Historical Node: YoY Revenue {'+' if yoy_rev >= 0 else ''}{yoy_rev:.1f}% | QoQ Revenue {'+' if qoq_rev >= 0 else ''}{qoq_rev:.1f}%."]
    }

# ── Node 5: Solvency Benchmark ──────────────────────────────────

def solvency_node(state: NexusState) -> Dict:
    """Altman Z''-Score for private firms: Z = 6.56X1 + 3.26X2 + 6.72X3 + 1.05X4"""
    acct = state['accounting']
    sales = state['sales']

    working_capital = acct['cash_balance'] + acct['total_ar'] - acct['total_ap']
    ta = acct['total_assets']
    tl = acct['total_liabilities']
    re = acct['retained_earnings']
    ebit = acct['gross_profit']  # Approximation: gross_profit ≈ EBIT for service firm
    eq = acct['equity']

    x1 = working_capital / ta  # Working Capital / Total Assets
    x2 = re / ta               # Retained Earnings / Total Assets
    x3 = ebit / ta             # EBIT / Total Assets
    x4 = eq / tl               # Equity / Total Liabilities

    z = (6.56 * x1) + (3.26 * x2) + (6.72 * x3) + (1.05 * x4)
    z = round(z, 2)

    if z >= 2.99:
        zone, color = "SAFE", "GREEN"
    elif z >= 1.81:
        zone, color = "GREY", "YELLOW"
    else:
        zone, color = "DISTRESS", "RED"

    # Sensitivity: Stress test scenarios
    scenarios = [
        {"label": "AR Default 30%", "ar_mult": 0.7, "exp_mult": 1.0},
        {"label": "AR Default 50%", "ar_mult": 0.5, "exp_mult": 1.0},
        {"label": "Expense Surge +30%", "ar_mult": 1.0, "exp_mult": 1.3},
        {"label": "Revenue Drop 20%", "ar_mult": 0.8, "exp_mult": 1.0},
        {"label": "Combined Stress", "ar_mult": 0.6, "exp_mult": 1.3},
        {"label": "Best Case: Full AR Recovery", "ar_mult": 1.3, "exp_mult": 0.9},
    ]

    sensitivity = []
    for s in scenarios:
        adj_ar = acct['total_ar'] * s['ar_mult']
        adj_wc = acct['cash_balance'] + adj_ar - acct['total_ap']
        adj_ebit = ebit / s['exp_mult']
        sz = (6.56 * (adj_wc / ta)) + (3.26 * x2) + (6.72 * (adj_ebit / ta)) + (1.05 * x4)
        sz = round(sz, 2)
        sensitivity.append({
            "label": s["label"],
            "z_score": sz,
            "zone": "Safe" if sz >= 2.99 else "Grey" if sz >= 1.81 else "Distress"
        })

    return {
        "solvency": {
            "z_score": z,
            "zone": zone,
            "color_code": color,
            "component_breakdown": {
                "X1_working_capital_ratio": round(x1, 4),
                "X2_retained_earnings_ratio": round(x2, 4),
                "X3_ebit_ratio": round(x3, 4),
                "X4_equity_leverage": round(x4, 4)
            },
            "sensitivity_scenarios": sensitivity
        },
        "audit_trail": [f"Solvency Node: Altman Z''-Score = {z} ({zone} Zone)."]
    }

# ── Node 6: Monte Carlo Cash Runway ─────────────────────────────

def monte_carlo_node(state: NexusState) -> Dict:
    """5000-iteration stochastic cash runway projection."""
    acct = state['accounting']
    sales = state['sales']

    # Calculate burn rate from expenses / months elapsed
    today = datetime.now()
    months_elapsed = max(today.month, 1)
    monthly_burn = max(acct['total_expenses'] / months_elapsed, 100)
    daily_burn = monthly_burn / 30

    cash = acct['cash_balance']
    ar = acct['total_ar']

    # Monte Carlo: Vary burn rate and AR recovery
    N = 5000
    sims = []
    for _ in range(N):
        v_burn = daily_burn * random.uniform(0.7, 1.5)
        v_recovery = ar * random.uniform(0.05, 0.95)
        runway_days = (cash + v_recovery) / max(v_burn, 0.01)
        sims.append(max(runway_days, 0))

    sims.sort()

    return {
        "cash_runway": {
            "burn_rate_monthly": round(monthly_burn, 2),
            "p5_days": int(sims[int(N * 0.05)]),
            "p50_days": int(sims[int(N * 0.50)]),
            "p95_days": int(sims[int(N * 0.95)]),
            "runway_months": round(sims[int(N * 0.50)] / 30, 1),
            "simulation_count": N
        },
        "audit_trail": [f"Monte Carlo Node: {N} simulations — P50 runway {int(sims[int(N * 0.50)])} days."]
    }

# ── Node 7: GPT-4-Turbo Strategic Oracle ────────────────────────

def strategic_oracle_node(state: NexusState) -> Dict:
    """Multi-Agent GPT-4-Turbo synthesis: Forensic Auditor + CFO + Synthesizer."""
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        logger.warning("OPENAI_API_KEY not set — using template strategic output")
        return {
            "strategic": {
                "key_insight": "OpenAI API key not configured. Add OPENAI_API_KEY env var for AI-powered strategic analysis.",
                "strategic_path_narrative": "Manual review required.",
                "strategic_path_bullets": ["Configure OPENAI_API_KEY to enable multi-agent analysis"],
                "immediate_actions": {"System": "Set OPENAI_API_KEY environment variable"},
                "impact_matrix": [],
                "risk_register": []
            },
            "audit_trail": ["Oracle Node: Skipped (no API key)."]
        }

    llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.1)

    # Build comprehensive context from ALL domains
    ctx = json.dumps({
        "sales": {
            "revenue": state['sales']['total_revenue'],
            "paid": state['sales']['paid_revenue'],
            "collection_rate": state['sales']['collection_rate'],
            "orders": state['sales']['total_orders'],
            "presale_funnel": state['sales']['presale_funnel'],
            "platform_breakdown": state['sales']['platform_breakdown']
        },
        "operations": {
            "total": state['operations']['total_operations'],
            "delivered_value": state['operations']['delivered_value'],
            "delivery_rate": state['operations']['delivery_rate']
        },
        "kpi": {
            "achievement_rate": state['kpi']['achievement_rate'],
            "bonus_paid_bdt": state['kpi']['total_bonus_paid_bdt'],
            "penalties": state['kpi']['penalty_count'],
            "top_performers": state['kpi']['top_performers'][:3],
            "underperformers": state['kpi']['underperformers'][:3]
        },
        "accounting": {
            "gross_profit": state['accounting']['gross_profit'],
            "gross_margin": state['accounting']['gross_margin'],
            "cash": state['accounting']['cash_balance'],
            "ar": state['accounting']['total_ar'],
            "ap": state['accounting']['total_ap'],
            "aging": state['accounting']['aging_ar']
        },
        "solvency": {
            "z_score": state['solvency']['z_score'],
            "zone": state['solvency']['zone'],
            "stress_tests": state['solvency']['sensitivity_scenarios']
        },
        "cash_runway": {
            "burn_rate": state['cash_runway']['burn_rate_monthly'],
            "p5_days": state['cash_runway']['p5_days'],
            "p50_days": state['cash_runway']['p50_days'],
            "p95_days": state['cash_runway']['p95_days']
        },
        "currency": {
            "usd_to_bdt": state['currency'].get('usd_to_bdt_rate', 120),
            "bonus_usd": state['currency'].get('bonus_total_usd', 0),
            "bonus_to_revenue_pct": state['currency'].get('bonus_to_revenue_pct', 0),
            "sales_bdt": state['currency'].get('sales_revenue_bdt', 0)
        },
        "historical_trends": {
            "yoy_revenue_growth": state['historical'].get('yoy_revenue_growth', 0),
            "yoy_collection_delta": state['historical'].get('yoy_collection_rate_delta', 0),
            "qoq_revenue_growth": state['historical'].get('qoq_revenue_growth', 0),
            "trend_signals": state['historical'].get('trend_signals', [])
        }
    }, indent=2, default=str)

    # AGENT 1: Forensic Auditor
    auditor_resp = llm.invoke([
        SystemMessage(content=(
            "You are a Senior Forensic Auditor for a Bangladeshi IT outsourcing firm (Betopia Group). "
            "Focus on: AR aging risks, collection rate anomalies, shadow payroll, KPI manipulation, "
            "platform concentration risk (Fiverr/Upwork dependency), and bonus-to-revenue ratios."
        )),
        HumanMessage(content=f"Audit this live data and identify TOP 5 forensic concerns:\n{ctx}")
    ]).content

    # AGENT 2: Strategic CFO
    cfo_resp = llm.invoke([
        SystemMessage(content=(
            "You are the CFO/Financialist for Betopia Group. The Forensic Auditor found these issues. "
            "Focus on: cash velocity, survival strategies, KPI structure optimization, "
            "platform diversification, and bonus expense management."
        )),
        HumanMessage(content=f"Auditor flags: {auditor_resp}\n\nFull metrics:\n{ctx}\n\nProvide specific survival actions.")
    ]).content

    # SYNTHESIZER: Boardroom-Ready Output
    synth_resp = llm.invoke([
        SystemMessage(content=(
            "Synthesize the Auditor and CFO perspectives into a JSON object for a professional sovereign verdict report. "
            "Use precise, boardroom-ready language specific to Betopia Group's actual numbers. "
            "Keys: key_insight (str), strategic_path_narrative (str), strategic_path_bullets (list of strings), "
            "immediate_actions (dict with keys: Sales, Operations, KPI, Finance, Legal), "
            "impact_matrix (list of dicts with keys: action, z_impact, cash_impact, risk), "
            "risk_register (list of dicts with keys: risk, probability, impact, mitigation)."
        )),
        HumanMessage(content=f"Auditor: {auditor_resp}\n\nCFO: {cfo_resp}\n\nMetrics:\n{ctx}")
    ], response_format={"type": "json_object"}).content

    try:
        analysis = json.loads(synth_resp)
    except Exception:
        analysis = {
            "key_insight": "AI synthesis parsing failed — review auditor output manually.",
            "strategic_path_narrative": cfo_resp[:500],
            "strategic_path_bullets": [],
            "immediate_actions": {},
            "impact_matrix": [],
            "risk_register": []
        }

    return {
        "strategic": analysis,
        "audit_trail": ["Oracle Node: Multi-agent GPT-4-Turbo synthesis complete."]
    }

# ── Node 8: Conflict Validation ─────────────────────────────────

def conflict_validation_node(state: NexusState) -> Dict:
    """Detect operational paradoxes: Legal vs Operations, KPI vs P&L."""
    conflicts = []
    directives = []

    # Conflict 1: AR collection vs operational dependency
    top_customers = state['sales'].get('top_customers', [])
    for cust in top_customers[:3]:
        if cust.get('total', 0) > state['sales']['total_revenue'] * 0.2:
            conflicts.append(
                f"CONCENTRATION RISK: {cust['name']} represents "
                f"${cust['total']:,.2f} ({cust['total']/max(state['sales']['total_revenue'],1)*100:.0f}% of revenue). "
                f"Legal recovery would disrupt operations."
            )
            directives.append({
                "id": len(directives) + 1,
                "type": "LEGAL",
                "target": cust['name'],
                "desc": f"AR Recovery: ${cust['total']:,.2f}"
            })

    # Conflict 2: Bonus expense vs profitability (currency-normalized)
    bonus_total = state['kpi']['total_bonus_paid_bdt']
    cx = state.get('currency', {})
    bonus_usd = cx.get('bonus_total_usd', 0) if cx else 0
    bonus_to_rev = cx.get('bonus_to_revenue_pct', 0) if cx else 0
    if bonus_total > 0 and state['accounting']['gross_profit'] > 0:
        bonus_ratio = bonus_total / max(state['accounting']['gross_profit'], 1) * 100
        if bonus_ratio > 30 or bonus_to_rev > 5:
            conflicts.append(
                f"BONUS PARADOX: KPI bonuses BDT {bonus_total:,.0f} (${bonus_usd:,.2f} USD) = "
                f"{bonus_to_rev:.1f}% of revenue. {'Unsustainable.' if bonus_to_rev > 10 else 'Monitor closely.'}"
            )

    # Conflict 3: High delivery rate but low collection
    if state['operations']['delivery_rate'] > 80 and state['sales']['collection_rate'] < 60:
        conflicts.append(
            f"DELIVERY-COLLECTION GAP: {state['operations']['delivery_rate']}% delivered "
            f"but only {state['sales']['collection_rate']}% collected. Revenue leakage risk."
        )

    # Conflict 4: KPI penalties vs retention
    if state['kpi']['penalty_count'] > state['kpi']['total_employees_tracked'] * 0.3:
        conflicts.append(
            f"KPI PENALTY CRISIS: {state['kpi']['penalty_count']} employees penalized "
            f"({state['kpi']['penalty_count']/max(state['kpi']['total_employees_tracked'],1)*100:.0f}%). "
            f"Retention risk."
        )

    # Standard strategic directives
    directives.extend([
        {"id": len(directives) + 1, "type": "FINANCE", "target": "ALL",
         "desc": f"AR Collection Drive (${state['accounting']['total_ar']:,.2f} outstanding)"},
        {"id": len(directives) + 1, "type": "KPI", "target": "HR",
         "desc": f"Review {state['kpi']['penalty_count']} penalty cases for Q{(datetime.now().month-1)//3+1}"},
        {"id": len(directives) + 1, "type": "OPS", "target": "DELIVERY",
         "desc": f"Clear ${state['operations']['pending_value']:,.2f} pending operations"}
    ])

    return {
        "proposed_directives": directives,
        "active_conflicts": conflicts,
        "status": "VALIDATING",
        "audit_trail": [f"Conflict Node: {len(conflicts)} strategic contradictions identified."]
    }

# ── Node 9: Authorization Gate ───────────────────────────────────

def authorization_gate_node(state: NexusState) -> Dict:
    """Sovereign Gate: Human-in-the-loop approval."""
    print("\n" + "█" * 70)
    print(" ODIN NEXUS v19.2: SOVEREIGN STRATEGIC ADVISOR GATEWAY")
    print("█" * 70)
    print(f"\n{'─'*70}")
    print(f" SOLVENCY: Z-Score {state['solvency']['z_score']} ({state['solvency']['zone']})")
    print(f" REVENUE:  ${state['sales']['total_revenue']:,.2f} | Collected: {state['sales']['collection_rate']}%")
    print(f" P&L:      Gross Profit ${state['accounting']['gross_profit']:,.2f} ({state['accounting']['gross_margin']}%)")
    print(f" RUNWAY:   {state['cash_runway']['p5_days']}–{state['cash_runway']['p95_days']} days | Burn ${state['cash_runway']['burn_rate_monthly']:,.2f}/mo")
    print(f" KPI:      {state['kpi']['achievement_rate']}% achievement | {state['kpi']['penalty_count']} penalties")
    cx = state.get('currency', {})
    if cx.get('usd_to_bdt_rate'):
        print(f" FX RATE:  1 USD = ৳{cx['usd_to_bdt_rate']:,.2f} BDT | Bonus ${cx.get('bonus_total_usd', 0):,.2f} ({cx.get('bonus_to_revenue_pct', 0):.1f}% of rev)")
    hist = state.get('historical', {})
    if hist.get('yoy_revenue_growth') is not None:
        yoy = hist['yoy_revenue_growth']
        qoq = hist.get('qoq_revenue_growth', 0)
        print(f" TRENDS:   YoY Revenue {'+' if yoy >= 0 else ''}{yoy:.1f}% | QoQ {'+' if qoq >= 0 else ''}{qoq:.1f}%")
    print(f"{'─'*70}")

    if state['active_conflicts']:
        print("\n⚠ ACTIVE CONFLICTS:")
        for c in state['active_conflicts']:
            print(f"  • {c}")

    approved = []
    for d in state["proposed_directives"]:
        choice = input(f"\n  APPROVE [{d['type']}] → {d['target']}: {d['desc']}? (y/n): ").lower().strip()
        if choice == 'y':
            approved.append(d["id"])

    return {
        "authorized_advisory_ids": approved,
        "status": "FINALIZED",
        "audit_trail": [f"Governance Gate: {len(approved)}/{len(state['proposed_directives'])} strategies authorized."]
    }

# ════════════════════════════════════════════════════════════════
# 4. SOVEREIGN VERDICT PDF ENGINE
# ════════════════════════════════════════════════════════════════
    print(f" REVENUE:  ${state['sales']['total_revenue']:,.2f} | Collected: {state['sales']['collection_rate']}%")
class SovereignVerdictPDF:
    """10-section executive report with full business flow coverage."""

    C_NAVY = colors.HexColor("#000080")
    C_DARK = colors.HexColor("#1a1a2e")
    C_RED = colors.HexColor("#c0392b")
    C_GREEN = colors.HexColor("#27ae60")
    C_AMBER = colors.HexColor("#f39c12")
    C_LIGHT = colors.HexColor("#f8f9fa")

    @classmethod
    def render(cls, state: NexusState) -> str:
        fpath = f"ODIN_SOVEREIGN_VERDICT_{state['report_date']}.pdf"
        doc = SimpleDocTemplate(fpath, pagesize=letter, leftMargin=0.5*inch, rightMargin=0.5*inch, topMargin=0.5*inch, bottomMargin=0.5*inch)
        styles = getSampleStyleSheet()
        story = []

        # Custom styles
        title_s = ParagraphStyle('Title', parent=styles['Heading1'], fontSize=22, textColor=cls.C_NAVY, alignment=TA_CENTER, spaceAfter=5)
        sub_s = ParagraphStyle('Sub', parent=styles['Italic'], fontSize=9, alignment=TA_CENTER, textColor=colors.grey)
        h2_s = ParagraphStyle('H2', parent=styles['Heading2'], fontSize=14, textColor=cls.C_NAVY, spaceBefore=15, spaceAfter=8)
        h3_s = ParagraphStyle('H3', parent=styles['Heading3'], fontSize=11, textColor=cls.C_DARK, spaceBefore=10, spaceAfter=5)
        body_s = ParagraphStyle('Body', parent=styles['Normal'], fontSize=9, leading=12)
        small_s = ParagraphStyle('Small', parent=styles['Normal'], fontSize=8, leading=10, textColor=colors.grey)
        hdr_s = ParagraphStyle('Hdr', parent=styles['Normal'], fontSize=8, fontName='Helvetica-Bold', textColor=colors.white, alignment=TA_CENTER)
        cell_s = ParagraphStyle('Cell', parent=styles['Normal'], fontSize=8, leading=10, alignment=TA_CENTER)
        cell_l = ParagraphStyle('CellL', parent=styles['Normal'], fontSize=8, leading=10)

        def make_table(headers, rows, col_widths=None):
            data = [[Paragraph(h, hdr_s) for h in headers]]
            for row in rows:
                data.append([Paragraph(str(c), cell_s) if not isinstance(c, Paragraph) else c for c in row])
            cw = col_widths or [7.5 * inch / len(headers)] * len(headers)
            t = Table(data, colWidths=cw)
            t.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), cls.C_NAVY),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
                ('GRID', (0, 0), (-1, -1), 0.5, colors.lightgrey),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, cls.C_LIGHT]),
                ('FONTSIZE', (0, 0), (-1, -1), 8),
            ]))
            return t

        z = state['solvency']['z_score']
        zone = state['solvency']['zone']
        llm = state.get('strategic', {})

        # ── HEADER ──────────────────────────────────────────────
        story.append(Paragraph("ODIN NEXUS v19.2: SOVEREIGN VERDICT", title_s))
        story.append(Paragraph(f"BETOPIA GROUP | STRICT READ-ONLY / ADVISORY MODE | {state['report_date']}", sub_s))
        story.append(Spacer(1, 10))

        # KEY INSIGHT BOX
        insight = llm.get('key_insight', 'Strategic analysis pending.')
        ki_style = ParagraphStyle('KI', backColor=cls.C_LIGHT, borderPadding=8, borderWidth=1, borderColor=cls.C_NAVY, fontSize=10, leading=13)
        story.append(Paragraph(f"<b>KEY INSIGHT:</b> {insight}", ki_style))
        story.append(Spacer(1, 8))

        # VERDICT BAR
        v_color = cls.C_GREEN if z >= 2.99 else cls.C_RED
        v_text = f"VERDICT: {'STABLE' if z >= 2.99 else 'GREY ZONE' if z >= 1.81 else 'FINANCIAL DISTRESS'} | Z-Score: {z}"
        v_style = ParagraphStyle('V', fontSize=12, textColor=colors.white, backColor=v_color, alignment=TA_CENTER, fontName='Helvetica-Bold', borderPadding=6)
        story.append(Paragraph(v_text, v_style))
        story.append(Spacer(1, 15))

        # ── I. EXECUTIVE SCORECARD ──────────────────────────────
        story.append(Paragraph("I. EXECUTIVE SCORECARD", h2_s))
        scorecard_data = [
            ["Total Revenue", f"${state['sales']['total_revenue']:,.2f}", f"{state['sales']['total_orders']} Orders"],
            ["Paid Revenue", f"${state['sales']['paid_revenue']:,.2f}", f"{state['sales']['collection_rate']}% Collection"],
            ["Gross Profit", f"${state['accounting']['gross_profit']:,.2f}", f"{state['accounting']['gross_margin']}% Margin"],
            ["Cash Balance", f"${state['accounting']['cash_balance']:,.2f}", f"Runway: {state['cash_runway']['runway_months']} mo"],
            ["Outstanding AR", f"${state['accounting']['total_ar']:,.2f}", f"AP: ${state['accounting']['total_ap']:,.2f}"],
            ["Operations Delivered", f"${state['operations']['delivered_value']:,.2f}", f"{state['operations']['delivery_rate']}% Rate"],
            ["KPI Achievement", f"{state['kpi']['achievement_rate']}%", f"{state['kpi']['total_employees_tracked']} Tracked"],
            ["Total Bonus (BDT)", f"BDT {state['kpi']['total_bonus_paid_bdt']:,.0f}", f"${state.get('currency', {}).get('bonus_total_usd', 0):,.2f} USD | {state['kpi']['penalty_count']} Penalties"],
            ["Altman Z-Score", str(z), zone],
            ["YoY Revenue", f"{'+' if state.get('historical', {}).get('yoy_revenue_growth', 0) >= 0 else ''}{state.get('historical', {}).get('yoy_revenue_growth', 0):.1f}%", f"QoQ: {'+' if state.get('historical', {}).get('qoq_revenue_growth', 0) >= 0 else ''}{state.get('historical', {}).get('qoq_revenue_growth', 0):.1f}%"],
            ["FX Rate", f"1 USD = ৳{state.get('currency', {}).get('usd_to_bdt_rate', 120):,.2f}", f"Bonus/Revenue: {state.get('currency', {}).get('bonus_to_revenue_pct', 0):.2f}%"],
        ]
        story.append(make_table(["Metric", "Value", "Context"], scorecard_data, [2.5*inch, 2.5*inch, 2.5*inch]))
        story.append(Spacer(1, 15))
        ki_style = ParagraphStyle('KI', backColor=cls.C_LIGHT, borderPadding=8, borderWidth=1, borderColor=cls.C_NAVY, fontSize=10, leading=13)
        # ── II. SALES PIPELINE ──────────────────────────────────
        story.append(Paragraph("II. SALES PIPELINE ANALYSIS", h2_s))

        # Top employees
        if state['sales']['top_employees']:
            story.append(Paragraph("Top Sales Performers", h3_s))
            emp_rows = []
            for e in state['sales']['top_employees'][:8]:
                emp_rows.append([
                    Paragraph(e['name'], cell_l),
                    f"${e['total']:,.2f}",
                    f"${e['paid']:,.2f}",
                    str(e['count']),
                    f"{e['paid']/max(e['total'],1)*100:.0f}%"
                ])
            story.append(make_table(["Employee", "Total Revenue", "Paid", "Orders", "Collection %"], emp_rows, [2*inch, 1.5*inch, 1.5*inch, 1*inch, 1.5*inch]))
            story.append(Spacer(1, 10))

        # Platform breakdown
        if state['sales']['platform_breakdown']:
            story.append(Paragraph("Revenue by Platform Source", h3_s))
            plat_rows = [[k, f"${v:,.2f}"] for k, v in sorted(state['sales']['platform_breakdown'].items(), key=lambda x: x[1], reverse=True)]
            story.append(make_table(["Platform", "Revenue (USD)"], plat_rows, [4*inch, 3.5*inch]))
            story.append(Spacer(1, 10))

        # Presale funnel
        pf = state['sales'].get('presale_funnel', {})
        if pf:
            story.append(Paragraph("Pre-Sale Conversion Funnel", h3_s))
            story.append(Paragraph(
                f"Total Queries: {pf.get('total_queries', 0)} → Converted: {pf.get('converted', 0)} → "
                f"Lost: {pf.get('lost', 0)} | <b>Conversion Rate: {pf.get('conversion_rate', 0)}%</b>",
                body_s
            ))
        story.append(Spacer(1, 10))
        Paragraph(e['name'], cell_l),
        # ── III. OPERATIONS DELIVERY ────────────────────────────
        story.append(Paragraph("III. OPERATIONS & DELIVERY PERFORMANCE", h2_s))

        # Status distribution
        if state['operations']['status_distribution']:
            story.append(Paragraph("Operation Status Distribution", h3_s))
            status_rows = [[k.upper(), str(v)] for k, v in state['operations']['status_distribution'].items()]
            story.append(make_table(["Status", "Count"], status_rows, [4*inch, 3.5*inch]))
            story.append(Spacer(1, 10))

        # Top delivery employees
        if state['operations']['top_delivery_employees']:
            story.append(Paragraph("Top Delivery Performers", h3_s))
            del_rows = []
            for e in state['operations']['top_delivery_employees'][:8]:
                del_rows.append([Paragraph(e['name'], cell_l), f"${e['delivered_value']:,.2f}", str(e['count'])])
            story.append(make_table(["Employee", "Delivered Value", "Operations"], del_rows, [3*inch, 2.5*inch, 2*inch]))
        story.append(Spacer(1, 10))

        # ── IV. KPI & BONUS ANALYSIS ────────────────────────────
        story.append(PageBreak())
        story.append(Paragraph("IV. KPI & BONUS ENGINE ANALYSIS", h2_s))

        # Grade distribution
        if state['kpi']['grade_distribution']:
            story.append(Paragraph("Grade Distribution", h3_s))
            grade_rows = [[k, str(v)] for k, v in state['kpi']['grade_distribution'].items()]
            story.append(make_table(["Grade", "Employees"], grade_rows, [4*inch, 3.5*inch]))
            story.append(Spacer(1, 10))

        # Top performers
        if state['kpi']['top_performers']:
            story.append(Paragraph("Top KPI Performers", h3_s))
            perf_rows = []
            for p in state['kpi']['top_performers'][:8]:
                perf_rows.append([
                    Paragraph(p['name'], cell_l),
                    p.get('role', ''),
                    f"${p['total_paid']:,.2f}",
                    f"${p['target']:,.2f}",
                    f"BDT {p['bonus']:,.0f}",
                    f"{'✓' if p['total_paid'] >= p['target'] else '✗'}"
                ])
            story.append(make_table(
                ["Employee", "Role", "Total Paid (USD)", "Target", "Bonus (BDT)", "Met?"],
                perf_rows, [1.8*inch, 0.8*inch, 1.3*inch, 1.1*inch, 1.2*inch, 0.6*inch]
            ))
            story.append(Spacer(1, 10))

        # Underperformers
        if state['kpi']['underperformers']:
            story.append(Paragraph("Underperformers (Below Target)", h3_s))
            under_rows = []
            for p in state['kpi']['underperformers'][:5]:
                shortfall = p['target'] - p['total_paid']
                under_rows.append([
                    Paragraph(p['name'], cell_l),
                    f"${p['total_paid']:,.2f}",
                    f"${p['target']:,.2f}",
                    Paragraph(f"<font color='red'>-${shortfall:,.2f}</font>", cell_s)
                ])
            story.append(make_table(["Employee", "Paid", "Target", "Shortfall"], under_rows, [2.5*inch, 1.5*inch, 1.5*inch, 2*inch]))
        story.append(Spacer(1, 10))
        p.get('role', ''),
        # ── V. ACCOUNTING & P&L ─────────────────────────────────
        story.append(Paragraph("V. ACCOUNTING & PROFIT/LOSS", h2_s))

        pnl_data = [
            ["Revenue Recognized", f"${state['accounting']['total_revenue_recognized']:,.2f}", "From posted invoices"],
            ["Total Expenses", f"${state['accounting']['total_expenses']:,.2f}", "From posted bills"],
            ["Gross Profit", f"${state['accounting']['gross_profit']:,.2f}", f"{state['accounting']['gross_margin']}% margin"],
            ["Total Assets", f"${state['accounting']['total_assets']:,.2f}", "Balance sheet"],
            ["Total Liabilities", f"${state['accounting']['total_liabilities']:,.2f}", ""],
            ["Equity", f"${state['accounting']['equity']:,.2f}", ""],
            ["Retained Earnings", f"${state['accounting']['retained_earnings']:,.2f}", ""],
        ]
        story.append(make_table(["Line Item", "Amount", "Notes"], pnl_data, [2.5*inch, 2.5*inch, 2.5*inch]))
        story.append(Spacer(1, 10))

        # AR Aging
        aging = state['accounting'].get('aging_ar', {})
        if aging:
            story.append(Paragraph("Accounts Receivable Aging", h3_s))
            aging_rows = [[k.replace('_', ' ').title(), f"${v:,.2f}"] for k, v in aging.items()]
            story.append(make_table(["Aging Bucket", "Amount (USD)"], aging_rows, [4*inch, 3.5*inch]))
            story.append(Spacer(1, 10))

        # Budget vs Actual
        if state['accounting']['budget_vs_actual']:
            story.append(Paragraph("Budget vs Actual", h3_s))
            bud_rows = []
            for b in state['accounting']['budget_vs_actual']:
                bud_rows.append([
                    Paragraph(b['category'], cell_l),
                    f"${b['planned']:,.2f}",
                    f"${b['actual']:,.2f}",
                    f"{b['variance_pct']:.1f}%"
                ])
            story.append(make_table(["Budget Category", "Planned", "Actual", "Variance %"], bud_rows, [2.5*inch, 1.5*inch, 1.5*inch, 2*inch]))
        story.append(Spacer(1, 15))

        # ── VI. MULTI-CURRENCY NORMALIZATION ────────────────────
        story.append(PageBreak())
        story.append(Paragraph("VI. MULTI-CURRENCY NORMALIZATION (USD ↔ BDT)", h2_s))

        cx = state.get('currency', {})
        if cx:
            fx_rate = cx.get('usd_to_bdt_rate', 120)
            story.append(Paragraph(
                f"<b>Live Exchange Rate:</b> 1 USD = ৳{fx_rate:,.2f} BDT (as of {cx.get('rate_date', 'N/A')})",
                body_s
            ))
            story.append(Spacer(1, 8))

            curr_data = [
                ["Sales Revenue", f"${cx.get('sales_revenue_usd', 0):,.2f}", f"৳{cx.get('sales_revenue_bdt', 0):,.0f}"],
                ["Gross Profit", f"${cx.get('gross_profit_usd', 0):,.2f}", f"৳{cx.get('gross_profit_bdt', 0):,.0f}"],
                ["KPI Bonuses", f"${cx.get('bonus_total_usd', 0):,.2f}", f"৳{cx.get('bonus_total_bdt', 0):,.0f}"],
                ["Bonus/Revenue %", f"{cx.get('bonus_to_revenue_pct', 0):.2f}%", "Normalized (USD basis)"],
            ]
            story.append(make_table(["Metric", "USD", "BDT"], curr_data, [2.5*inch, 2.5*inch, 2.5*inch]))
            story.append(Spacer(1, 10))

            # Currency breakdown by sale order currency
            cbreakdown = cx.get('currency_breakdown', [])
            if cbreakdown:
                story.append(Paragraph("Revenue by Order Currency", h3_s))
                cb_rows = [[b.get('currency', '?'), f"${b.get('total', 0):,.2f}", str(b.get('count', 0))] for b in cbreakdown]
                story.append(make_table(["Currency", "Total Amount", "Order Count"], cb_rows, [2.5*inch, 2.5*inch, 2.5*inch]))
            story.append(Spacer(1, 10))

        # ── VII. HISTORICAL TREND (YoY / QoQ) ──────────────────
        story.append(Paragraph("VII. HISTORICAL TREND ANALYSIS (YoY / QoQ)", h2_s))

        hist = state.get('historical', {})
        if hist:
            # Trend signals
            for sig in hist.get('trend_signals', []):
                sig_color = cls.C_RED if any(w in sig for w in ['DECLINE', 'COLLAPSE', 'SLUMP', 'DETERIORATION']) else cls.C_GREEN if 'GROWTH' in sig else cls.C_DARK
                story.append(Paragraph(f"<font color='{sig_color}'><b>▶ {sig}</b></font>", body_s))
            story.append(Spacer(1, 10))

            def _fmt_delta(v):
                sign = '+' if v >= 0 else ''
                return f"{sign}{v:.1f}%"

            yoy_data = [
                ["Revenue Growth", _fmt_delta(hist.get('yoy_revenue_growth', 0)), f"Prior Year: ${hist.get('prior_year_revenue', 0):,.2f}"],
                ["Order Volume", _fmt_delta(hist.get('yoy_orders_growth', 0)), f"Prior Year: {hist.get('prior_year_orders', 0)} orders"],
                ["Collection Rate Δ", f"{'+' if hist.get('yoy_collection_rate_delta', 0) >= 0 else ''}{hist.get('yoy_collection_rate_delta', 0):.1f}pp", "Year-over-Year change"],
                ["Delivery Rate Δ", f"{'+' if hist.get('yoy_delivery_rate_delta', 0) >= 0 else ''}{hist.get('yoy_delivery_rate_delta', 0):.1f}pp", "Year-over-Year change"],
                ["KPI Achievement Δ", f"{'+' if hist.get('yoy_kpi_achievement_delta', 0) >= 0 else ''}{hist.get('yoy_kpi_achievement_delta', 0):.1f}pp", "Year-over-Year change"],
            ]
            story.append(Paragraph("Year-over-Year Comparison", h3_s))
            story.append(make_table(["Metric", "YoY Change", "Context"], yoy_data, [2.5*inch, 2*inch, 3*inch]))
            story.append(Spacer(1, 10))

            qoq_data = [
                ["Revenue Growth", _fmt_delta(hist.get('qoq_revenue_growth', 0)), f"Prior Quarter: ${hist.get('prior_quarter_revenue', 0):,.2f}"],
                ["Order Volume", _fmt_delta(hist.get('qoq_orders_growth', 0)), f"Prior Quarter: {hist.get('prior_quarter_orders', 0)} orders"],
            ]
            story.append(Paragraph("Quarter-over-Quarter Comparison", h3_s))
            story.append(make_table(["Metric", "QoQ Change", "Context"], qoq_data, [2.5*inch, 2*inch, 3*inch]))
            story.append(Spacer(1, 10))

            # Quarterly trend table
            qt = hist.get('quarterly_trend', [])
            if qt:
                story.append(Paragraph("Quarterly Revenue Trend (Last 4 Quarters)", h3_s))
                qt_rows = [[q['quarter'], f"${q['revenue']:,.2f}", str(q['orders']), f"{q['collection_rate']}%"] for q in qt]
                story.append(make_table(["Quarter", "Revenue", "Orders", "Collection %"], qt_rows, [2*inch, 2*inch, 1.5*inch, 2*inch]))
        story.append(Spacer(1, 15))

        # ── VIII. SOLVENCY HEATMAP ────────────────────────────────
        story.append(PageBreak())
        story.append(Paragraph("VIII. SOLVENCY & RISK HEATMAP", h2_s))

        # Z-Score components
        components = state['solvency']['component_breakdown']
        comp_rows = [
            ["X1: Working Capital / Assets", str(components['X1_working_capital_ratio']), "Weight: 6.56"],
            ["X2: Retained Earnings / Assets", str(components['X2_retained_earnings_ratio']), "Weight: 3.26"],
            ["X3: EBIT / Assets", str(components['X3_ebit_ratio']), "Weight: 6.72"],
            ["X4: Equity / Liabilities", str(components['X4_equity_leverage']), "Weight: 1.05"],
        ]
        story.append(make_table(["Z-Score Component", "Ratio", "Altman Weight"], comp_rows, [3*inch, 2*inch, 2.5*inch]))
        story.append(Spacer(1, 10))

        # Sensitivity
        story.append(Paragraph("Stress Test Scenarios", h3_s))
        sens_rows = []
        for s in state['solvency']['sensitivity_scenarios']:
            z_color = cls.C_GREEN if s['z_score'] >= 2.99 else cls.C_AMBER if s['z_score'] >= 1.81 else cls.C_RED
            sens_rows.append([
                Paragraph(s['label'], cell_l),
                str(s['z_score']),
                Paragraph(f"<font color='{z_color}'>{s['zone']}</font>", cell_s)
            ])
        story.append(make_table(["Scenario", "Z-Score", "Outlook"], sens_rows, [3.5*inch, 1.5*inch, 2.5*inch]))
        story.append(Spacer(1, 15))

        # ── IX. CASH RUNWAY ────────────────────────────────────
        story.append(Paragraph("IX. MONTE CARLO CASH RUNWAY PROJECTION", h2_s))
        cr = state['cash_runway']
        runway_data = [
            ["Monthly Burn Rate", f"${cr['burn_rate_monthly']:,.2f}", "Average monthly expenses"],
            ["P5 Survival Floor", f"{cr['p5_days']} Days", "5th percentile — worst case"],
            ["P50 Median Runway", f"{cr['p50_days']} Days", "Expected outcome"],
            ["P95 Survival Ceiling", f"{cr['p95_days']} Days", "Best-case scenario"],
            ["Simulations", str(cr['simulation_count']), "Monte Carlo iterations"],
        ]
        story.append(make_table(["Metric", "Value", "Interpretation"], runway_data, [2.5*inch, 2*inch, 3*inch]))
        story.append(Spacer(1, 15))

        # ── X. STRATEGIC CONFLICTS ───────────────────────────
        if state['active_conflicts']:
            story.append(Paragraph("X. STRATEGIC CONFLICT REGISTER", h2_s))
            for i, c in enumerate(state['active_conflicts'], 1):
                story.append(Paragraph(f"<b>{i}.</b> <font color='red'>{c}</font>", body_s))
            story.append(Spacer(1, 15))
            Paragraph(s['label'], cell_l),
        # ── XI. STRATEGIC PATH & IMPACT MATRIX ──────────────────
        story.append(Paragraph("XI. RECOMMENDED STRATEGIC PATH", h2_s))

        narrative = llm.get('strategic_path_narrative', 'AI analysis pending.')
        story.append(Paragraph(narrative, body_s))
        story.append(Spacer(1, 8))

        for bullet in llm.get('strategic_path_bullets', []):
            story.append(Paragraph(f"• {bullet}", body_s))
        story.append(Spacer(1, 10))

        # Immediate Actions
        actions = llm.get('immediate_actions', {})
        if actions:
            story.append(Paragraph("Immediate Actions (Next 7 Days)", h3_s))
            for dept, action in actions.items():
                story.append(Paragraph(f"<b>{dept}:</b> {action}", body_s))
            story.append(Spacer(1, 10))

        # Impact Matrix
        im = llm.get('impact_matrix', [])
        if im:
            story.append(Paragraph("Impact Matrix", h3_s))
            im_rows = [[
                Paragraph(i.get('action', ''), cell_l),
                i.get('z_impact', ''),
                i.get('cash_impact', ''),
                i.get('risk', '')
            ] for i in im]
            story.append(make_table(["Action", "Z-Score Impact", "Cash Impact", "Risk"], im_rows, [2.2*inch, 1.5*inch, 2.3*inch, 1.5*inch]))
            story.append(Spacer(1, 10))

        # Risk Register
        rr = llm.get('risk_register', [])
        if rr:
            story.append(Paragraph("Risk Register", h3_s))
            rr_rows = [[
                Paragraph(r.get('risk', ''), cell_l),
                r.get('probability', ''),
                r.get('impact', ''),
                Paragraph(r.get('mitigation', ''), cell_l)
            ] for r in rr]
            story.append(make_table(["Risk", "Probability", "Impact", "Mitigation"], rr_rows, [2*inch, 1*inch, 1*inch, 3.5*inch]))
        story.append(Spacer(1, 15))

        # ── XII. DECISION TIMELINE & AUDIT TRAIL ──────────────────
        story.append(PageBreak())
        story.append(Paragraph("XII. DECISION TIMELINE", h2_s))
        deadline = (datetime.now() + timedelta(days=cr['p5_days'])).strftime('%Y-%m-%d')
        story.append(Paragraph(f"<b>DECISION DEADLINE: {deadline}</b> (Based on P5 survival floor of {cr['p5_days']} days)", body_s))
        story.append(Paragraph("Inaction beyond this date increases insolvency probability significantly.", body_s))
        story.append(Spacer(1, 20))

        story.append(Paragraph("IMMUTABLE SYSTEM AUDIT TRAIL", h2_s))
        for log in state['audit_trail']:
            story.append(Paragraph(f"> {log}", small_s))

        # Footer
        story.append(Spacer(1, 30))
        footer_s = ParagraphStyle('F', fontSize=7, textColor=colors.grey, alignment=TA_CENTER)
        story.append(HRFlowable(width="100%", color=colors.lightgrey))
        story.append(Paragraph(
            f"ODIN NEXUS v19.2 | Data Source: {state['data_source']} | "
            f"ADVISORY CONFIDENCE: 85% | MODEL FIDELITY: 92% | RISK OF INACTION: CRITICAL",
            footer_s
        ))

        doc.build(story)
        logger.info(f"PDF generated: {fpath}")
        return fpath

# ════════════════════════════════════════════════════════════════
# 5. GRAPH ORCHESTRATION
# ════════════════════════════════════════════════════════════════

def build_nexus():
    workflow = StateGraph(NexusState)

    # Register nodes (prefixed with 'n_' to avoid state key conflicts)
    workflow.add_node("n_sales", sales_pipeline_node)
    workflow.add_node("n_operations", operations_node)
    workflow.add_node("n_kpi", kpi_engine_node)
    workflow.add_node("n_accounting", accounting_node)
    workflow.add_node("n_currency", currency_normalization_node)
    workflow.add_node("n_historical", historical_trend_node)
    workflow.add_node("n_solvency", solvency_node)
    workflow.add_node("n_monte_carlo", monte_carlo_node)
    workflow.add_node("n_oracle", strategic_oracle_node)
    workflow.add_node("n_conflicts", conflict_validation_node)
    workflow.add_node("n_gateway", authorization_gate_node)

    # Topology: Data harvest → Currency/Historical → Analysis → Strategy → Governance
    workflow.set_entry_point("n_sales")
    workflow.add_edge("n_sales", "n_operations")
    workflow.add_edge("n_operations", "n_kpi")
    workflow.add_edge("n_kpi", "n_accounting")
    workflow.add_edge("n_accounting", "n_currency")
    workflow.add_edge("n_currency", "n_historical")
    workflow.add_edge("n_historical", "n_solvency")
    workflow.add_edge("n_solvency", "n_monte_carlo")
    workflow.add_edge("n_monte_carlo", "n_oracle")
    workflow.add_edge("n_oracle", "n_conflicts")
    workflow.add_edge("n_conflicts", "n_gateway")
    workflow.add_edge("n_gateway", END)

    return workflow.compile(checkpointer=MemorySaver())


# ════════════════════════════════════════════════════════════════
# 6. RUNTIME
# ════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("  ODIN NEXUS v19.2: SOVEREIGN MULTI-AGENT STRATEGIC ADVISOR")
    print("  Betopia Group | Full Business Flow Intelligence")
    print("━" * 70)

    init_state = {
        "session_metadata": {"id": "NEXUS-V19.2-PROD", "mode": "ADVISORY"},
        "sales": {}, "operations": {}, "kpi": {}, "accounting": {},
        "solvency": {}, "cash_runway": {}, "strategic": {},
        "currency": {}, "historical": {},
        "proposed_directives": [], "active_conflicts": [],
        "authorized_advisory_ids": [],
        "audit_trail": ["ODIN Nexus v19.2 Boot: Full Business Flow Intelligence Active."],
        "status": "SCANNING",
        "data_source": "INITIALIZING",
        "report_date": datetime.now().strftime('%Y-%m-%d')
    }

    app = build_nexus()
    thread = {"configurable": {"thread_id": "sovereign-v19.2"}}

    print("\nLaunching 11-node intelligence pipeline...")
    print("  [1] Sales Pipeline → [2] Operations → [3] KPI Engine")
    print("  [4] Accounting/P&L → [5] Currency Normalization → [6] Historical YoY/QoQ")
    print("  [7] Solvency Z-Score → [8] Monte Carlo → [9] GPT-4 Oracle")
    print("  [10] Conflict Detection → [11] Governance Gate\n")

    final_output = app.invoke(init_state, thread)

    # Generate Sovereign Verdict PDF
    report_path = SovereignVerdictPDF.render(final_output)

    print(f"\nSOVEREIGN VERDICT FINALIZED: {report_path}")
    print("━" * 70)    
    print(f"Audit Trail: {len(final_output['audit_trail'])} entries logged.")

    print(f"Data Source: {final_output['data_source']}")
    print(f"Data Source: {final_output['data_source']}")

    print(f"Audit Trail: {len(final_output['audit_trail'])} entries logged.")
    from datetime import datetime

# Initial state
init_state = {
    "session_metadata": {"id": "NEXUS-V19.2-PROD", "mode": "ADVISORY"},
    "sales": {},
    "operations": {},
    "kpi": {},
    "accounting": {},
    "solvency": {},
    "cash_runway": {},
    "strategic": {},
    "proposed_directives": [],
    "active_conflicts": [],
    "authorized_advisory_ids": [],
    "audit_trail": [
        "ODIN Nexus v19.2 Boot: Full Business Flow Intelligence Active."
    ],
    "status": "SCANNING",
    "data_source": "INITIALIZING",
    "report_date": datetime.now().strftime('%Y-%m-%d')
}

# Thread config
thread = {
    "configurable": {
        "thread_id": "sovereign-v19.2"
    }
}

# Build app
app = build_nexus()

# Pipeline logs (before execution)
print("\nLaunching 9-node intelligence pipeline...")
print("  [1] Sales Pipeline → [2] Operations → [3] KPI Engine")
print("  [4] Accounting/P&L → [5] Solvency Z-Score → [6] Monte Carlo")
print("  [7] GPT-4 Oracle → [8] Conflict Detection → [9] Governance Gate\n")

# Run pipeline
final_output = app.invoke(init_state, thread)

# Audit log
print(f"Audit Trail: {len(final_output['audit_trail'])} entries logged.")

# Generate PDF
report_path = SovereignVerdictPDF.render(final_output)

# Final message
print(f"\nSOVEREIGN VERDICT FINALIZED: {report_path}")